<a href="https://colab.research.google.com/github/robbarto2/GenAI-Foundations/blob/main/Section%206%20-%20Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Transfer Learning: Full Fine-Tuning

**Run on Colab with GPU runtime.** This notebook requires a GPU—use **Google Colab** and set the runtime to T4 GPU.

This notebook demonstrates **full fine-tuning** (transfer learning)—every parameter in the model is updated during training. This is the simplest approach but also the most expensive in terms of compute, memory, and time.

### Why 1B instead of 3B?

We use `Llama-3.2-1B-Instruct` here because **full fine-tuning of the 3B model runs out of memory (OOM) on a free Colab T4** (16 GB VRAM). With LoRA or Unsloth, only a fraction of parameters are trainable, so the 3B model fits comfortably. This is itself a key teaching point: full fine-tuning has much higher memory requirements.

### Three-notebook progression

| Notebook | Approach | Model | Trainable Params | Expected Time |
|---|---|---|---|---|
| **This notebook** | Full fine-tuning | 1B | 100% (~1.2B) | Longest |
| **LoRA** | LoRA adapters | 3B | ~1-2% | ~8 min |
| **LoRA + Unsloth** | Optimized LoRA | 3B | ~1-2% | ~3 min |

In [ ]:
!pip install --disable-pip-version-check transformers peft torch datasets accelerate -q

Import the necessary libraries

In [ ]:
import torch
import os
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import Dataset

os.environ["WANDB_DISABLED"] = "true"

Log into Hugging Face with a secure token

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("Hugging Face token not found in Colab secrets.")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Successfully logged into Hugging Face!")

Create Some Instruction Training Examples to Fine-Tune the Model

In [ ]:
examples = []

In [ ]:
examples = [
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I haven't received my order #12345 yet. It's been a week.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Delayed
Reference: #12345
Action Taken: Order status check initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I received the wrong item in my order #67890.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Wrong Item Delivered
Reference: #67890
Action Taken: Replacement item initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #98765 is damaged. What should I do?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Damaged Item
Reference: #98765
Action Taken: Return and replacement process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I want to cancel my order #54321.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Cancellation Request
Reference: #54321
Action Taken: Cancellation process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I was charged twice for my order #11223. Please help.",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Billing Issue
Reference: #11223
Action Taken: Refund request initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "When will my order #44556 arrive?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: In Transit
Reference: #44556
Action Taken: Estimated delivery date provided

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #99988 was returned. When will I get my refund?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Returned
Reference: #99988
Action Taken: Refund process initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I can't track my order #66777. Can you help?",
        "output": """Thank you for contacting TechCorp Support.

Order Status: Tracking Issue
Reference: #66777
Action Taken: Tracking link resent

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
TechCorp Support"""
    },
        {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #56789 was supposed to arrive yesterday, but I haven't received it.",
        "output": """Thank you for contacting Tech Support.

Order Status: Delayed
Reference: #56789
Action Taken: Initiated a follow-up with the shipping carrier

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I received a damaged item in my order #98765. What should I do?",
        "output": """Thank you for contacting Tech Support.

Order Status: Damaged Item
Reference: #98765
Action Taken: Replacement process initiated. Return instructions sent.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I was charged twice for my order #11223. Please help.",
        "output": """Thank you for contacting Tech Support.

Order Status: Billing Issue
Reference: #11223
Action Taken: Refund process initiated for duplicate charge

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "When will my order #33445 arrive?",
        "output": """Thank you for contacting Tech Support.

Order Status: In Transit
Reference: #33445
Action Taken: Estimated delivery date provided via email

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I need to cancel my order #44556 immediately.",
        "output": """Thank you for contacting Tech Support.

Order Status: Cancellation Requested
Reference: #44556
Action Taken: Cancellation process initiated. Confirmation email sent.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I can't track my order #77889. The tracking link doesn't work.",
        "output": """Thank you for contacting Tech Support.

Order Status: Tracking Issue
Reference: #77889
Action Taken: Sent updated tracking link. Contacted carrier for more information.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "I returned my order #99000 last week. When will I get my refund?",
        "output": """Thank you for contacting Tech Support.

Order Status: Returned
Reference: #99000
Action Taken: Refund process initiated. Refund expected within 5 business days.

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    },
    {
        "instruction": "Respond to this customer inquiry following our format",
        "input": "My order #11122 arrived, but it's missing some items.",
        "output": """Thank you for contacting Tech Support.

Order Status: Incomplete Order
Reference: #11122
Action Taken: Missing items identified and replacement initiated

Need immediate assistance? Call: 1-800-TECH-HELP
Your satisfaction is our priority.

Best regards,
Tech Support"""
    }
]

Format our training examples the way we want (Instruction, Input, Response)

In [ ]:
dataset = Dataset.from_list(examples)

In [ ]:
def format_example(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output = example["output"]
    return {
        "text": f"### Instruction: {instruction}\n### Input: {input_text}\n### Response: {output}"
    }

formatted_examples = [format_example(ex) for ex in examples]
dataset = Dataset.from_list(formatted_examples)

### Load the model

We load `Llama-3.2-1B-Instruct` in **float16** directly onto the GPU. Unlike the LoRA notebooks, there is **no 4-bit quantization** and **no LoRA adapters**—every parameter in the model will be updated during training.

This means higher VRAM usage, which is why we use the 1B model instead of 3B.

In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"
max_seq_length = 512
hf_token = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token,
)

print(f"Model loaded: {model_name}")
print(f"Device: {model.device}")

In [ ]:
# Ensure tokenizer has a padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Split into train/validation (80/20), then tokenize
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

def tokenize_fn(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_dataset = split_dataset["train"].map(tokenize_fn).remove_columns(["text"])
eval_dataset = split_dataset["test"].map(tokenize_fn).remove_columns(["text"])
print(f"Train: {len(train_dataset)} examples, Eval: {len(eval_dataset)} examples")
print(f"Columns: {train_dataset.column_names}")

In [ ]:
# All parameters are trainable (no LoRA, no freezing)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.1f}%")

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=30,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    fp16=True,
    report_to="none",
)

In [ ]:
# Check VRAM before training
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits 2>/dev/null || echo "nvidia-smi not available"

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [ ]:
train_start = time.time()
trainer.train()
train_elapsed = (time.time() - train_start) / 60

trainer.save_model("./final_model")
print("Training complete and model saved!")
print(f"Full fine-tuning took {train_elapsed:.1f} min.")
print(f"Compare: LoRA ~8 min, LoRA + Unsloth ~3 min (on the 3B model).")

### Inference

Test the fine-tuned model on a customer inquiry.

In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """
    Generate a response using the fine-tuned model.

    Args:
        model: The fine-tuned model
        tokenizer: The tokenizer
        instruction: The instruction for the model
        input_text: Optional input text

    Returns:
        str: The generated response
    """
    # Format input to match training format
    prompt = f"### Instruction: {instruction}\n### Input: {input_text}\n### Response:"

    # Tokenize the prompt
    inputs = tokenizer(prompt,
                      return_tensors="pt",
                      truncation=True,
                      max_length=512,
                      add_special_tokens=True)

    # Move inputs to the same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        num_return_sequences=1,
        temperature=0.1,
        do_sample=True,
        top_p=0.95,
        top_k=50,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Decode the response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response part (after "### Response:")
    response_parts = response.split("### Response:")
    if len(response_parts) > 1:
        response = response_parts[1].strip()

    return response

In [ ]:
def post_process_response(response):
    """
    Post-process the model's response to ensure it adheres to the desired format.

    Args:
        response (str): The raw response from the model.

    Returns:
        str: The post-processed response with required sections.
    """
    # Define required sections
    sections = ["Order Status", "Reference", "Action Taken", "Need immediate assistance?"]
    processed_response = response.strip()

    # Ensure all required sections exist
    for section in sections:
        if section not in processed_response:
            processed_response += f"\n{section}: [Details Missing]"

    return processed_response

In [ ]:
# Example input
instruction = "Respond to this customer inquiry following our format"
input_text = "I haven't received my order #12345 yet. It's been a week."

raw_response = generate_response(model, tokenizer, instruction, input_text)
final_response = post_process_response(raw_response)

# Print the results
print("\nINSTRUCTION:")
print(instruction)
print("\nINPUT:")
print(input_text)
print("\nFINAL RESPONSE:")
print(final_response)

In [ ]:
# Check training logs/loss
print("Training logs from the last few steps:")
print(trainer.state.log_history[-5:])